<a href="https://colab.research.google.com/github/koli440/ipo-evaluator/blob/main/IPO_evaluator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.5 MB/s eta 0:00:00


In [30]:
from datetime import datetime
import feedparser  # Knihovna pro čtení RSS/Atom feedů
import requests
from email.mime.multipart import MIMEMultipart
from email.header import Header
from email.mime.text import MIMEText
import smtplib
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')


def send_email_report(analysis_text):
  sender_email = userdata.get('emailAddress')
  receiver_email = userdata.get('emailAddress')
  app_password = userdata.get('gmailAppPassword')

  msg = MIMEText(analysis_text, "html", "utf-8")
  msg["From"] = sender_email
  msg["To"] = receiver_email

  subject_text = (
      f"IPO Radar - Denní přehled ({datetime.now().strftime('%Y-%m-%d')})"
  )
  msg["Subject"] = Header(subject_text, "utf-8")

  try:
    server = smtplib.SMTP("smtp.gmail.com", 587)
    server.starttls()
    server.login(sender_email, app_password)
    server.sendmail(sender_email, receiver_email, msg.as_string())
    server.quit()
    print("Email byl úspěšně odeslán na tvůj Gmail v HTML formátu.")
  except Exception as e:
    print(f"Chyba při odesílání emailu: {e}")


def fetch_ipo_rss_news():
  # Veřejné RSS kanály a zdroje pro sledování nových emisí a tiskových zpráv
  rss_urls = [
    "https://finance.yahoo.com/rss/headline?s=IPO",
    "https://www.benzinga.com/feed",
    "https://seekingalpha.com/market_currents.xml",
    "https://www.cnbc.com/id/10000664/device/rss/rss.html",
]

  articles = []

  # Pro zkušební účely si můžeme pomoci i přímým vyhledáním čerstvých zpráv přes public endpointy
  # Zkusíme stáhnout aktuální RSS z Yahoo Finance / NASDAQ
  headers = {
      "User-Agent": (
          "Mozilla/5.0 (Windows NT 10.6; Win64; x64) AppleWebKit/537.36"
      )
  }

  # Alternativně využijeme Yahoo Finance RSS pro IPO
  yahoo_ipo_rss = "https://finance.yahoo.com/rss/headline?s=IPO"

  for url in rss_urls:
    feed = feedparser.parse(url)
    print(url)
    for entry in feed.entries:
        title = entry.get("title", "")
        summary = entry.get("summary", "")
        published = entry.get("published", "")

        # Filtrujeme klíčová slova spojená s IPO
        if "IPO" in title or "public offering" in title.lower():
          articles.append(
              {"title": title, "summary": summary, "published": published}
          )

  return articles


def analyze_with_gemini(ipo_data):
  if not ipo_data:
    return "V RSS feedech nebyly nalezeny žádné čerstvé zprávy o IPO."

  prompt = f"""
    Jsi analytický bot pro sledování akciového trhu. Dnes je {datetime.now().strftime('%Y-%m-%d')}.
    Zde jsou nejnovější zprávy a tiskové zprávy o IPO získané z veřejných RSS zdrojů:
    {ipo_data}

    Analyzuj je a odpověz v čistém HTML formátu (použij tagy jako <b>, <ul>, <li>, žádný markdown):
    1. Název společnosti (pokud je z textu zřejmý)
    2. O co se jedná
    3. Zda zpráva indikuje reálný potenciál pro krátkodobý momentum trading.
    """

  url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent?key={GEMINI_API_KEY}"
  headers = {"Content-Type": "application/json"}
  payload = {"contents": [{"parts": [{"text": prompt}]}]}

  response = requests.post(url, json=payload, headers=headers)
  if response.status_code == 200:
    result = response.json()
    return result["candidates"][0]["content"]["parts"][0]["text"]
  else:
    return f"Chyba při volání Gemini API: {response.text}"


# Na konec hlavní části skriptu přidej:
if __name__ == "__main__":
  print("Stahuji čerstvá data o IPO přes RSS...")
  ipo_news = fetch_ipo_rss_news()
  print(f"Nalezeno {len(ipo_news)} článků. Spouštím analýzu...")
  analysis = analyze_with_gemini(ipo_news)

  print("\n--- VÝSLEDEK ANALÝZY ---")
  print(analysis)

  # Odeslání výsledku na mail
  send_email_report(analysis)

Stahuji čerstvá data o IPO přes RSS...
https://finance.yahoo.com/rss/headline?s=IPO
https://www.benzinga.com/feed
https://seekingalpha.com/market_currents.xml
https://www.cnbc.com/id/10000664/device/rss/rss.html
Nalezeno 6 článků. Spouštím analýzu...

--- VÝSLEDEK ANALÝZY ---
<div>
    <h3>Analýza IPO trhu k 14. 9. 2026</h3>
    
    <ul>
        <li>
            <b>1. SpaceX (SPCX)</b><br>
            <b>O co se jedná:</b> Společnost realizovala historické mega-IPO, při kterém vybrala přibližně 75 až 86 miliard USD, čímž úspěšně otevřela IPO okno pro druhou polovinu roku 2026.<br>
            <b>Potenciál pro krátkodobý momentum trading:</b> <b>Střední až vysoký.</b> Přestože prvotní debutový denní skok proběhl na přelomu června a srpna, masivní likvidita, rebalancování fondů a následné zprávy stále vytvářejí výraznou volatilitu vhodnou pro swingové i intradenní obchodování.
        </li>
        <br>
        <li>
            <b>2. OpenAI</b><br>
            <b>O co se jedná:</b> Spol